# Phase 1 — Credit Risk Scoring (WOE + Logistic Regression)

Clean linear pipeline: load raw → source aggregates → modelling frame + feature taxonomy → WOE binning & IV selection → WOE logistic regression → score & export. Run top to bottom; no cell depends on out-of-order state.

In [6]:
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

import joblib, json, sys, sklearn
from datetime import date

import warnings
warnings.filterwarnings('ignore')

RAW = Path('data/raw')
OUT = Path('data/processed')
OUT.mkdir(parents=True, exist_ok=True)

## 1. Load raw data

In [2]:

app = pd.read_csv(RAW / 'application_train.csv')
prev = pd.read_csv(RAW / 'previous_application.csv')

bureau = pd.read_csv(RAW / 'bureau.csv')
bb = pd.read_csv(RAW / 'bureau_balance.csv')

print(f'application_train : {app.shape}')
print(f'previous application : {prev.shape}')
print(f'bureau            : {bureau.shape}')
print(f'Target rate       : {app.TARGET.mean():.3%}')

application_train : (307511, 122)
previous application : (1670214, 37)
bureau            : (1716428, 17)
Target rate       : 8.073%


## 2. Source aggregates (bureau / previous_application / bureau_balance)

In [3]:
con = duckdb.connect()
con.register('bureau', bureau)

bureau_agg = con.execute("""
    SELECT
        SK_ID_CURR,

        -- volume
        COUNT(*)                                        AS bur_n_credits,
        COUNT(*) FILTER (WHERE CREDIT_ACTIVE = 'Active') AS bur_n_active,
        COUNT(*) FILTER (WHERE CREDIT_ACTIVE = 'Closed') AS bur_n_closed,

        -- overdue flags
        SUM(AMT_CREDIT_MAX_OVERDUE)                     AS bur_total_max_overdue,
        AVG(AMT_CREDIT_MAX_OVERDUE)                     AS bur_avg_max_overdue,
        SUM(CNT_CREDIT_PROLONG)                         AS bur_total_prolonged,
        AVG(CREDIT_DAY_OVERDUE)                         AS bur_avg_days_overdue,
        MAX(CREDIT_DAY_OVERDUE)                         AS bur_max_days_overdue,

        -- debt load
        SUM(AMT_CREDIT_SUM)                             AS bur_total_credit,
        SUM(AMT_CREDIT_SUM_DEBT)                        AS bur_total_debt,
        AVG(AMT_CREDIT_SUM_DEBT)                        AS bur_avg_debt,
        SUM(AMT_CREDIT_SUM_OVERDUE)                     AS bur_total_overdue_amt,
        AVG(COALESCE(AMT_CREDIT_SUM_DEBT, 0)
            / NULLIF(AMT_CREDIT_SUM, 0))                AS bur_avg_util_ratio,

        -- recency
        MIN(DAYS_CREDIT)                                AS bur_oldest_credit_days,
        MAX(DAYS_CREDIT)                                AS bur_newest_credit_days,
        AVG(DAYS_CREDIT)                                AS bur_avg_credit_days,
        AVG(DAYS_CREDIT_ENDDATE)                        AS bur_avg_end_days,

        -- active sub-aggregates
        SUM(AMT_CREDIT_SUM_DEBT)
            FILTER (WHERE CREDIT_ACTIVE = 'Active')     AS bur_active_debt,
        AVG(AMT_CREDIT_SUM_DEBT)
            FILTER (WHERE CREDIT_ACTIVE = 'Active')     AS bur_active_avg_debt

    FROM bureau
    GROUP BY SK_ID_CURR
""").df()

print(f'bureau_agg: {bureau_agg.shape}')
bureau_agg.head(3)

bureau_agg: (305811, 20)


,SK_ID_CURR,bur_n_credits,bur_n_active,bur_n_closed,bur_total_max_overdue,bur_avg_max_overdue,bur_total_prolonged,bur_avg_days_overdue,bur_max_days_overdue,bur_total_credit,bur_total_debt,bur_avg_debt,bur_total_overdue_amt,bur_avg_util_ratio,bur_oldest_credit_days,bur_newest_credit_days,bur_avg_credit_days,bur_avg_end_days,bur_active_debt,bur_active_avg_debt
0,357485,4,2,2,23535.00,23535.00,0.0,0.0,0,1964302.290,313713.0,104571.0,0.0,0.088095,-2711,-251,-1577.750000,-335.000000,313713.0,156856.5
1,323031,7,0,7,0.00,0.00,0.0,0.0,0,621733.500,0.0,0.0,0.0,0.000000,-2026,-252,-1230.857143,-1013.285714,NaN,NaN
2,144364,7,1,6,4814.82,802.47,0.0,0.0,0,371659.905,0.0,0.0,0.0,0.000000,-2916,-1045,-2265.142857,-1915.428571,0.0,0.0


In [4]:
# Previous-application aggregates.
# DAYS_DECISION is negative (days before application); max = least negative = most recent.
prev_agg = prev.groupby('SK_ID_CURR').agg(
    prev_n_applications     = ('SK_ID_PREV', 'count'),
    prev_days_last_decision = ('DAYS_DECISION', 'max'),
    prev_n_approved         = ('NAME_CONTRACT_STATUS', lambda x: (x == 'Approved').sum()),
).reset_index()

prev_agg['prev_days_last_decision'] = prev_agg['prev_days_last_decision'].abs()
print(f'prev_agg: {prev_agg.shape}')

prev_agg: (338857, 4)


In [5]:
bb['STATUS_NUM'] = pd.to_numeric(bb['STATUS'], errors='coerce').fillna(0)

# Level 1: per credit account (SK_ID_BUREAU)
bb_bureau = bb.groupby('SK_ID_BUREAU').agg(
    bb_months_observed = ('MONTHS_BALANCE', 'count'),
    bb_max_dpd         = ('STATUS_NUM', 'max'),
    bb_n_months_bad    = ('STATUS_NUM', lambda x: (x > 0).sum()),
).reset_index()

bb_bureau['bb_has_any_dpd'] = (bb_bureau['bb_max_dpd'] > 0).astype(int)

# Map SK_ID_BUREAU → SK_ID_CURR via raw bureau
bb_with_curr = bureau.merge(bb_bureau, on='SK_ID_BUREAU', how='left')

# Level 2: per applicant (SK_ID_CURR)
bb_agg = bb_with_curr.groupby('SK_ID_CURR').agg(
    bb_max_dpd             = ('bb_max_dpd', 'max'),
    bb_total_months_bad    = ('bb_n_months_bad', 'sum'),
    bb_n_credits_with_dpd  = ('bb_has_any_dpd', 'sum'),
    bb_total_months_obs    = ('bb_months_observed', 'sum'),
).reset_index()

bb_agg['bb_pct_months_bad'] = (
    bb_agg['bb_total_months_bad'] / bb_agg['bb_total_months_obs']
)


## 3. Application pre-processing & feature group taxonomy

Single source of truth for the feature→group taxonomy (`FEATURE_TO_GROUP`). Every downstream artifact inherits its `feature_group` tag from here.

In [6]:
# === Pre-process application columns ===
app_work = app.copy()

# DAYS_EMPLOYED sentinel — 365243 ≈ 1000 years, means "unemployed"
# (positive value where every other "days" column is negative). NaN it.
app_work.loc[app_work['DAYS_EMPLOYED'] == 365243, 'DAYS_EMPLOYED'] = np.nan

# Engineered financial ratios — safe division returns NaN on non-positive denominator
def _safe_div(numer, denom):
    return np.where(denom > 0, numer / denom, np.nan)

app_work['annuity_to_income'] = _safe_div(app_work['AMT_ANNUITY'],    app_work['AMT_INCOME_TOTAL'])
app_work['credit_to_income']  = _safe_div(app_work['AMT_CREDIT'],     app_work['AMT_INCOME_TOTAL'])
app_work['credit_to_goods']   = _safe_div(app_work['AMT_CREDIT'],     app_work['AMT_GOODS_PRICE'])

# === Feature group taxonomy ===
FEATURE_GROUPS = {
    # --- From application_train.csv (some engineered) ---
    'demographic': [
        'DAYS_BIRTH', 'CODE_GENDER', 'NAME_EDUCATION_TYPE',
        'NAME_FAMILY_STATUS', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS',
    ],
    'employment': [
        'DAYS_EMPLOYED', 'NAME_INCOME_TYPE', 'ORGANIZATION_TYPE', 'OCCUPATION_TYPE',
    ],
    'financial': [
        'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
        'annuity_to_income', 'credit_to_income', 'credit_to_goods',
    ],
    'housing': [
        'NAME_HOUSING_TYPE', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
    ],
    'region': [
        'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY', 'REGION_POPULATION_RELATIVE',
    ],
    'identity_recency': [
        'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'DAYS_LAST_PHONE_CHANGE',
    ],
    'bureau_enquiry': [
        'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_YEAR',
    ],
    
    # --- From bureau.csv aggregates (Section 2) ---
    'bureau_credit': [
        'bur_n_credits', 'bur_n_active', 'bur_n_closed',
        'bur_total_credit', 'bur_total_debt', 'bur_avg_debt',
        'bur_avg_util_ratio', 'bur_active_debt', 'bur_active_avg_debt',
        'bur_oldest_credit_days', 'bur_newest_credit_days',
        'bur_avg_credit_days', 'bur_avg_end_days',
        'has_bureau_history',
    ],
    'bureau_delinquency': [
        'bur_total_max_overdue', 'bur_avg_max_overdue',
        'bur_total_prolonged', 'bur_avg_days_overdue', 'bur_max_days_overdue',
        'bur_total_overdue_amt',
        # bureau_balance.csv aggregates:
        'bb_max_dpd', 'bb_total_months_bad', 'bb_n_credits_with_dpd',
        'bb_total_months_obs', 'bb_pct_months_bad',
    ],
    
    # --- From previous_application.csv aggregates ---
    'previous_application': [
        'prev_n_applications', 'prev_days_last_decision',
        'prev_n_approved', 'has_previous_application',
    ],
}

# Derived lookups
FEATURE_TO_GROUP = {f: g for g, feats in FEATURE_GROUPS.items() for f in feats}
CANDIDATE_FEATURES = list(FEATURE_TO_GROUP.keys())

# Application-derived subset (everything we pull from app_work; aggregates are joined separately)
_APP_DERIVED_GROUPS = {'demographic', 'employment', 'financial', 'housing',
                       'region', 'identity_recency', 'bureau_enquiry'}
APP_FEATURE_COLS = [f for g, feats in FEATURE_GROUPS.items()
                    if g in _APP_DERIVED_GROUPS for f in feats]

# Sanity print
print(f"Feature groups: {len(FEATURE_GROUPS)}")
print(f"Total candidate features: {len(CANDIDATE_FEATURES)}")
print(f"  - application-derived: {len(APP_FEATURE_COLS)}")
print(f"  - aggregate-derived:   {len(CANDIDATE_FEATURES) - len(APP_FEATURE_COLS)}")
print()
for g, feats in FEATURE_GROUPS.items():
    print(f"  {g:25s} {len(feats):3d}")

Feature groups: 10
Total candidate features: 57
  - application-derived: 28
  - aggregate-derived:   29

  demographic                 6
  employment                  4
  financial                   7
  housing                     3
  region                      3
  identity_recency            3
  bureau_enquiry              2
  bureau_credit              14
  bureau_delinquency         11
  previous_application        4


In [7]:
META = ['SK_ID_CURR', 'TARGET']

# Widened modelling dataset:
#   - app-level features come pre-processed from app_work
#   - aggregates from bureau / previous_application / bureau_balance (Section 2 + above)
df = (
    app_work[META + APP_FEATURE_COLS]
    .merge(bureau_agg, on='SK_ID_CURR', how='left')
    .merge(prev_agg,   on='SK_ID_CURR', how='left')
    .merge(bb_agg,     on='SK_ID_CURR', how='left')
)

# --- Aggregate fillna logic (preserved from v1) ---
# bb_agg NaN = no bureau records to compute DPD on → 0 is the natural fill
bb_cols = ['bb_max_dpd', 'bb_total_months_bad', 'bb_n_credits_with_dpd',
           'bb_total_months_obs', 'bb_pct_months_bad']
df[bb_cols] = df[bb_cols].fillna(0)

# bureau_agg NaN = no bureau history → 0 counts, indicator flag preserved
df['bur_n_credits']      = df['bur_n_credits'].fillna(0)
df['has_bureau_history'] = (df['bur_n_credits'] > 0).astype(int)
df['bur_n_active']       = df['bur_n_active'].fillna(0)
df['bur_n_closed']       = df['bur_n_closed'].fillna(0)

# prev_agg NaN = no previous applications with this lender
df['has_previous_application']  = df['prev_n_applications'].notna().astype(int)
df['prev_n_applications']       = df['prev_n_applications'].fillna(0)
df['prev_n_approved']           = df['prev_n_approved'].fillna(0)
df['prev_days_last_decision']   = df['prev_days_last_decision'].fillna(
    df['prev_days_last_decision'].median()
)

# Application-derived features keep their NaN — WOE binning handles missing
# as exclusion-from-bin at training time and WOE = 0 at apply time.

# Sanity check — every CANDIDATE_FEATURE should now exist in df
missing = [f for f in CANDIDATE_FEATURES if f not in df.columns]
if missing:
    print(f"!!! MISSING from df: {missing}")
else:
    print(f"All {len(CANDIDATE_FEATURES)} candidate features present in df")

print(f"\nDataset: {df.shape}")
print(f"\nMissing rate — top 10 columns:")
print(df[CANDIDATE_FEATURES].isna().mean().sort_values(ascending=False).head(10).to_string())

All 57 candidate features present in df

Dataset: (307511, 59)

Missing rate — top 10 columns:
bur_total_max_overdue    0.402018
bur_avg_max_overdue      0.402018
bur_active_debt          0.328434
bur_active_avg_debt      0.328434
OCCUPATION_TYPE          0.313455
DAYS_EMPLOYED            0.180072
bur_total_debt           0.167083
bur_avg_debt             0.167083
bur_avg_end_days         0.150463
bur_avg_util_ratio       0.146671


In [8]:
# === Rare-category collapse (decision 8) — slots into Section 3, before the IV loop ===
MIN_CATEGORY_FRAC = 0.01   # levels below ~1% of non-missing rows -> "Other"

# categorical candidates, detected the same way process_feature does
cat_candidates = [
    f for f in CANDIDATE_FEATURES
    if not (pd.api.types.is_numeric_dtype(df[f]) and df[f].nunique(dropna=True) > 5)
]

def collapse_rare(s, min_frac):
    vc = s.value_counts(normalize=True)          # fraction of NON-missing rows
    keep = set(vc[vc >= min_frac].index)
    if len(keep) == s.nunique(dropna=True):      # nothing rare -> leave untouched
        return s, keep
    return s.where(s.isin(keep) | s.isna(), other="Other"), keep

RARE_KEEP, changed = {}, []                       # RARE_KEEP persisted for scoring (step 7)
for feat in cat_candidates:
    before = df[feat].nunique(dropna=True)
    df[feat], RARE_KEEP[feat] = collapse_rare(df[feat], MIN_CATEGORY_FRAC)
    after = df[feat].nunique(dropna=True)
    if after != before:
        changed.append(feat)
        print(f"{feat:<28} {before:>3} -> {after:>3} levels "
              f"(collapsed {before-after} rare into 'Other')")
if not changed:
    print("no rare levels found at this threshold")

NAME_INCOME_TYPE               8 ->   5 levels (collapsed 3 rare into 'Other')
ORGANIZATION_TYPE             58 ->  17 levels (collapsed 41 rare into 'Other')
OCCUPATION_TYPE               18 ->  13 levels (collapsed 5 rare into 'Other')
NAME_HOUSING_TYPE              6 ->   5 levels (collapsed 1 rare into 'Other')


## 4. WOE binning & IV feature selection

In [9]:
# === WOE binning utilities ===

def _woe_iv_from_groups(grouped, total_events, total_non_events, smoothing=0.5):
    """Laplace-smoothed WOE and IV contribution per bin."""
    n_bins = len(grouped)
    dist_e  = (grouped['n_events']     + smoothing) / (total_events     + smoothing * n_bins)
    dist_ne = (grouped['n_non_events'] + smoothing) / (total_non_events + smoothing * n_bins)
    woe = np.log(dist_e / dist_ne)
    iv_contrib = (dist_e - dist_ne) * woe
    return woe.values, iv_contrib.values


def _stats_from_bins(bin_series, y):
    """Group y by bin assignment; preserves bin order."""
    df = pd.DataFrame({'bin': bin_series, 'y': y.values}).dropna(subset=['bin'])
    g = (df.groupby('bin', observed=True, sort=True)
           .agg(n_obs=('y', 'count'), n_events=('y', 'sum'))
           .reset_index())
    g['n_non_events'] = g['n_obs'] - g['n_events']
    g['event_rate']   = g['n_events'] / g['n_obs']
    return g

In [10]:
## Monotomic Binning
def find_monotonic_bins_numeric(x, y, n_bins_initial=10, min_bin_share=0.05):
    """
    Return bin edges for a numeric feature with monotonic event rate.
    
    For zero-rich features (>50% values ≤ 0), the 0 edge is pinned: zero gets
    its own bin with independently floating WOE, and monotonicity is enforced
    only across the positive tail. This preserves IV signal that would otherwise
    be destroyed by qcut producing duplicate edges at 0.
    
    Returns edge list with -inf and +inf as outer bounds, or None if the
    feature has too little variation to bin.
    """
    mask = x.notna()
    x_clean, y_clean = x[mask], y[mask]
    
    if x_clean.nunique() < 2 or len(x_clean) < n_bins_initial * 5:
        return None
    
    # Detect zero-rich and bin the positive tail separately
    zero_pct = (x_clean <= 0).mean()
    is_zero_rich = zero_pct > 0.5
    pinned_edge_idx = None

    if is_zero_rich:
        x_positive = x_clean[x_clean > 0]
        if x_positive.nunique() < 2 or len(x_positive) < n_bins_initial * 2:
            # >50% of values are <= 0 but there's no usable positive tail.
            # This isn't a "floor at zero" feature — it's a fully non-positive
            # continuous axis (e.g. DAYS_* offsets stored as negatives).
            # Demote to ordinary binning instead of silently dropping it.
            is_zero_rich = False
        else:
            try:
                _, pos_edges = pd.qcut(x_positive, q=n_bins_initial,
                                        retbins=True, duplicates='drop')
            except ValueError:
                return None
            if len(pos_edges) < 2:
                return None
            interior = [float(e) for e in pos_edges[1:-1]]
            edges = [-np.inf, 0.0] + interior + [np.inf]
            pinned_edge_idx = 1  # the 0.0 edge — never drop

    if not is_zero_rich:
        try:
            _, edges_arr = pd.qcut(x_clean, q=n_bins_initial,
                                    retbins=True, duplicates='drop')
        except ValueError:
            return None
        if len(edges_arr) < 3:
            return None
        edges = list(edges_arr)
        edges[0]  = -np.inf
        edges[-1] =  np.inf    
    total_e  = y_clean.sum()
    total_ne = (1 - y_clean).sum()
    
    def stats_for(edge_list):
        b = pd.cut(x_clean, bins=edge_list, include_lowest=True)
        s = _stats_from_bins(b, y_clean)
        woe, iv = _woe_iv_from_groups(s, total_e, total_ne)
        s['woe'] = woe
        s['iv_contrib'] = iv
        return s
    
    def droppable_edge_indices():
        """Interior edge indices that can be dropped (skip outer edges and pinned)."""
        return [i for i in range(1, len(edges) - 1) if i != pinned_edge_idx]
    
    # Step 1: enforce minimum bin share
    # For zero-rich features, share is measured within the non-zero tail
    # (otherwise a 5% global minimum is unsatisfiable when non-zero is <50% of data).
    while len(edges) > 3:
        s = stats_for(edges)
        n_obs = s['n_obs'].values
        
        if is_zero_rich:
            nonzero_obs = n_obs[1:]
            if nonzero_obs.sum() == 0 or len(nonzero_obs) == 0:
                break
            nonzero_shares = nonzero_obs / nonzero_obs.sum()
            if nonzero_shares.min() >= min_bin_share:
                break
            smallest_bin = 1 + int(np.argmin(nonzero_shares))
        else:
            shares = n_obs / n_obs.sum()
            if shares.min() >= min_bin_share:
                break
            smallest_bin = int(np.argmin(shares))
        
        # Pick the merge that loses least IV
        candidates = []
        if smallest_bin > 0 and smallest_bin != pinned_edge_idx:
            candidates.append(smallest_bin)
        if (smallest_bin + 1) < (len(edges) - 1) and (smallest_bin + 1) != pinned_edge_idx:
            candidates.append(smallest_bin + 1)
        if not candidates:
            break
        
        current_iv = s['iv_contrib'].sum()
        best_loss, best_drop = np.inf, candidates[0]
        for cand in candidates:
            trial = edges[:cand] + edges[cand+1:]
            trial_iv = stats_for(trial)['iv_contrib'].sum()
            loss = current_iv - trial_iv
            if loss < best_loss:
                best_loss, best_drop = loss, cand
        edges.pop(best_drop)
    
    # Step 2: enforce monotonicity (across non-zero bins only when pinned)
    while len(edges) > 3:
        s = stats_for(edges)
        rates = s['event_rate'].values
        # When zero bin is pinned, check monotonicity only on the non-zero tail
        rates_to_check = rates[1:] if is_zero_rich else rates
        if len(rates_to_check) < 2:
            break
        diffs = np.diff(rates_to_check)
        if np.all(diffs >= 0) or np.all(diffs <= 0):
            break
        
        current_iv = s['iv_contrib'].sum()
        best_loss, best_drop = np.inf, None
        for j in droppable_edge_indices():
            trial = edges[:j] + edges[j+1:]
            trial_iv = stats_for(trial)['iv_contrib'].sum()
            loss = current_iv - trial_iv
            if loss < best_loss:
                best_loss, best_drop = loss, j
        if best_drop is None:
            break
        edges.pop(best_drop)
    
    return edges

In [11]:
def process_feature(name, x, y, feature_type='auto',
                    n_bins_initial=10, min_bin_share=0.05):
    """
    Bin one feature and return its WOE table in canonical schema.
    Output columns:
      feature_name, feature_type, bin_index, bin_label,
      lower_bound, upper_bound, category_value,
      n_obs, n_events, n_non_events, event_rate, woe, iv_contribution
    Total IV is stored as df.attrs['iv_total'].
    Returns None if the feature can't be binned.
    """
    if feature_type == 'auto':
        feature_type = 'numeric' if (
            pd.api.types.is_numeric_dtype(x) and x.nunique(dropna=True) > 5
        ) else 'categorical'
    
    mask = x.notna()
    if mask.sum() == 0:
        return None
    
    x_m, y_m = x[mask], y[mask]
    total_e  = y_m.sum()
    total_ne = (1 - y_m).sum()
    
    if feature_type == 'numeric':
        edges = find_monotonic_bins_numeric(x, y, n_bins_initial, min_bin_share)
        if edges is None:
            return None
        bin_series = pd.cut(x_m, bins=edges, include_lowest=True)
        stats = _stats_from_bins(bin_series, y_m)
        woe, iv = _woe_iv_from_groups(stats, total_e, total_ne)
        stats['woe'] = woe
        stats['iv_contribution'] = iv
        
        out = pd.DataFrame({
            'feature_name':    name,
            'feature_type':    'numeric',
            'bin_index':       range(len(stats)),
            'bin_label':       stats['bin'].astype(str).values,
            'lower_bound':     [iv_.left  for iv_ in stats['bin']],
            'upper_bound':     [iv_.right for iv_ in stats['bin']],
            'category_value':  None,
            'n_obs':           stats['n_obs'].values,
            'n_events':        stats['n_events'].values,
            'n_non_events':    stats['n_non_events'].values,
            'event_rate':      stats['event_rate'].values,
            'woe':             stats['woe'].values,
            'iv_contribution': stats['iv_contribution'].values,
        })
    else:
        x_str = x_m.astype(str)
        stats = _stats_from_bins(x_str, y_m)
        woe, iv = _woe_iv_from_groups(stats, total_e, total_ne)
        stats['woe'] = woe
        stats['iv_contribution'] = iv
        stats = stats.sort_values('event_rate').reset_index(drop=True)
        
        out = pd.DataFrame({
            'feature_name':    name,
            'feature_type':    'categorical',
            'bin_index':       range(len(stats)),
            'bin_label':       stats['bin'].astype(str).values,
            'lower_bound':     None,
            'upper_bound':     None,
            'category_value':  stats['bin'].astype(str).values,
            'n_obs':           stats['n_obs'].values,
            'n_events':        stats['n_events'].values,
            'n_non_events':    stats['n_non_events'].values,
            'event_rate':      stats['event_rate'].values,
            'woe':             stats['woe'].values,
            'iv_contribution': stats['iv_contribution'].values,
        })
    
    out.attrs['iv_total'] = float(out['iv_contribution'].sum())
    return out

In [12]:
# Sanity check (optional): WOE table for two features.
# Test 1: a numeric v1 feature
tbl = process_feature('bur_total_debt', df['bur_total_debt'], df['TARGET'])
print(f"--- bur_total_debt | IV = {tbl.attrs['iv_total']:.4f} ---")
print(tbl[['bin_label', 'n_obs', 'event_rate', 'woe', 'iv_contribution']].to_string(index=False))
print()

# Test 2: another numeric, this one was IV-strong in v1
tbl = process_feature('bb_pct_months_bad', df['bb_pct_months_bad'], df['TARGET'])
print(f"--- bb_pct_months_bad | IV = {tbl.attrs['iv_total']:.4f} ---")
print(tbl[['bin_label', 'n_obs', 'event_rate', 'woe', 'iv_contribution']].to_string(index=False))

--- bur_total_debt | IV = 0.0430 ---
           bin_label  n_obs  event_rate       woe  iv_contribution
         (-inf, 0.0]  70985    0.055603 -0.356769         0.030377
     (0.0, 184500.0]  57083    0.079358  0.024416         0.000134
(184500.0, 324310.5]  25612    0.083594  0.081135         0.000681
     (324310.5, inf] 102451    0.090346  0.166042         0.011830

--- bb_pct_months_bad | IV = 0.0145 ---
         bin_label  n_obs  event_rate       woe  iv_contribution
       (-inf, 0.0] 276459    0.078612 -0.029052         0.000750
    (0.0, 0.00466]   3107    0.066302 -0.210399         0.000410
(0.00466, 0.00775]   3118    0.077614 -0.041044         0.000017
 (0.00775, 0.0116]   3114    0.079640 -0.013121         0.000002
  (0.0116, 0.0169]   3084    0.088521  0.102112         0.000109
  (0.0169, 0.0236]   3111    0.092896  0.155060         0.000260
  (0.0236, 0.0333]   3178    0.099434  0.230156         0.000603
  (0.0333, 0.0473]   3024    0.106151  0.302978         0.001025
  

In [13]:
# === Section 4 (rebuilt): WOE binning across the widened candidate pool ===
# Produces:
#   bin_definitions — long table, one row per (feature, bin), tagged with feature_group
#   iv_ranking      — one row per feature: iv_total + binning diagnostics
# Assumes df (widened frame), CANDIDATE_FEATURES, FEATURE_TO_GROUP in scope.

y = df['TARGET'] 

bin_tables, iv_rows, skipped = [], [], []

for feat in CANDIDATE_FEATURES:
    tbl = process_feature(feat, df[feat], y, feature_type='auto')
    if tbl is None:
        skipped.append(feat)
        continue

    tbl['feature_group'] = FEATURE_TO_GROUP[feat]
    bin_tables.append(tbl)

    is_numeric   = tbl['feature_type'].iloc[0] == 'numeric'
    has_zero_pin = bool(
        is_numeric and
        ((tbl['lower_bound'] == -np.inf) & (tbl['upper_bound'] == 0.0)).any()
    )
    iv_rows.append({
        'feature_name':  feat,
        'feature_group': FEATURE_TO_GROUP[feat],
        'feature_type':  tbl['feature_type'].iloc[0],
        'missing_rate':  round(float(df[feat].isna().mean()), 3),
        'n_bins':        len(tbl),
        'has_zero_pin':  has_zero_pin,
        'iv_total':      round(float(tbl.attrs['iv_total']), 4),
    })

bin_definitions = pd.concat(bin_tables, ignore_index=True)
iv_ranking = (pd.DataFrame(iv_rows)
                .sort_values('iv_total', ascending=False)
                .reset_index(drop=True))

print(f"binned {len(iv_rows)} / {len(CANDIDATE_FEATURES)} features "
      f"({len(skipped)} skipped)")
if skipped:
    print("skipped (process_feature returned None):", skipped)

# conventional IV bands for quick reading
def _band(iv):
    if iv < 0.02: return 'drop?'
    if iv < 0.10: return 'weak'
    if iv < 0.30: return 'medium'
    if iv < 0.50: return 'strong'
    return 'suspicious'   # >0.5 usually leakage / near-target proxy
iv_ranking['band'] = iv_ranking['iv_total'].map(_band)

with pd.option_context('display.max_rows', None):
    print(iv_ranking.to_string(index=False))

binned 57 / 57 features (0 skipped)
               feature_name        feature_group feature_type  missing_rate  n_bins  has_zero_pin  iv_total   band
         bur_avg_util_ratio        bureau_credit      numeric         0.147       4         False    0.1434 medium
        bur_avg_credit_days        bureau_credit      numeric         0.143       9         False    0.1322 medium
              DAYS_EMPLOYED           employment      numeric         0.180       8         False    0.0923   weak
                 DAYS_BIRTH          demographic      numeric         0.000      10         False    0.0842   weak
     bur_newest_credit_days        bureau_credit      numeric         0.143       5         False    0.0804   weak
            OCCUPATION_TYPE           employment  categorical         0.313      13         False    0.0792   weak
     bur_oldest_credit_days        bureau_credit      numeric         0.143       9         False    0.0768   weak
            credit_to_goods            finan

In [14]:
# === Step 4: aggressive correlation prune ===
# Assumes df, bin_definitions, iv_ranking are still in memory from the IV cell.
import numpy as np
import pandas as pd

def woe_transform(frame, bin_defs, features):
    """Map raw values to WOE via bin_definitions. Missing/unseen -> WOE 0."""
    out = pd.DataFrame(index=frame.index)
    for feat in features:
        d = bin_defs[bin_defs['feature_name'] == feat].sort_values('bin_index')
        if d['feature_type'].iloc[0] == 'numeric':
            edges = [d['lower_bound'].iloc[0]] + d['upper_bound'].tolist()
            idx = pd.cut(frame[feat], bins=edges, labels=False, include_lowest=True)
            woe_by_idx = d['woe'].to_numpy()
            mapped = pd.Series(0.0, index=frame.index)
            valid = idx.notna()
            mapped[valid] = woe_by_idx[idx[valid].astype(int).to_numpy()]
            out[feat] = mapped
        else:
            lut = dict(zip(d['category_value'].astype(str), d['woe']))
            out[feat] = frame[feat].astype(str).map(lut).fillna(0.0)
    return out

IV_FLOOR       = 0.02
CORR_THRESHOLD = 0.85
TOP_N          = 12

# candidates above the IV floor, in IV order
ranked = iv_ranking[iv_ranking['iv_total'] >= IV_FLOOR].sort_values(
    'iv_total', ascending=False)
candidates = ranked['feature_name'].tolist()

# WOE matrix + absolute correlation
woe_mat = woe_transform(df, bin_definitions, candidates)
corr = woe_mat.corr().abs()

# (1) greedy redundancy prune — keep higher-IV of each correlated pair
kept, dropped = [], []
for feat in candidates:
    clash = next((k for k in kept if corr.loc[feat, k] > CORR_THRESHOLD), None)
    if clash is None:
        kept.append(feat)
    else:
        dropped.append((feat, clash, round(float(corr.loc[feat, clash]), 3)))

# (2) hard cap to top-N by IV (kept is already IV-ordered)
selected = kept[:TOP_N]

print(f"candidates above IV {IV_FLOOR}: {len(candidates)}")
print(f"after corr prune (>{CORR_THRESHOLD}): {len(kept)} kept, {len(dropped)} dropped")
for f, against, c in dropped:
    print(f"  drop {f:<27} (corr {c} with {against})")
print(f"\nFINAL SELECTED ({len(selected)}):")
sel = iv_ranking.set_index('feature_name').loc[selected, ['feature_group', 'iv_total']]
print(sel.to_string())
print("\ngroup mix:")
print(sel['feature_group'].value_counts().to_string())

candidates above IV 0.02: 30
after corr prune (>0.85): 25 kept, 5 dropped
  drop bur_oldest_credit_days      (corr 0.873 with bur_avg_credit_days)
  drop REGION_RATING_CLIENT        (corr 0.954 with REGION_RATING_CLIENT_W_CITY)
  drop bur_total_debt              (corr 0.948 with bur_avg_debt)
  drop bur_total_max_overdue       (corr 0.972 with bur_avg_max_overdue)
  drop bur_active_avg_debt         (corr 0.95 with bur_active_debt)

FINAL SELECTED (12):
                             feature_group  iv_total
feature_name                                        
bur_avg_util_ratio           bureau_credit    0.1434
bur_avg_credit_days          bureau_credit    0.1322
DAYS_EMPLOYED                   employment    0.0923
DAYS_BIRTH                     demographic    0.0842
bur_newest_credit_days       bureau_credit    0.0804
OCCUPATION_TYPE                 employment    0.0792
credit_to_goods                  financial    0.0709
NAME_INCOME_TYPE                employment    0.0579
bur_avg_end_d

In [15]:
# Freeze the model feature set from the prune output — the single source of truth.
# Deliberately NOT hardcoded: it follows `selected` so the notebook stays honest if
# the data or thresholds ever change. Everything downstream uses SELECTED_FEATURES.
SELECTED_FEATURES = list(selected)
print(f"{len(SELECTED_FEATURES)} model features:")
for f in SELECTED_FEATURES:
    print(f"  {f:<32} {FEATURE_TO_GROUP[f]}")

12 model features:
  bur_avg_util_ratio               bureau_credit
  bur_avg_credit_days              bureau_credit
  DAYS_EMPLOYED                    employment
  DAYS_BIRTH                       demographic
  bur_newest_credit_days           bureau_credit
  OCCUPATION_TYPE                  employment
  credit_to_goods                  financial
  NAME_INCOME_TYPE                 employment
  bur_avg_end_days                 bureau_credit
  ORGANIZATION_TYPE                employment
  REGION_RATING_CLIENT_W_CITY      region
  NAME_EDUCATION_TYPE              demographic


## 5. Train logistic regression (WOE pipeline)

Pipeline is `WOEEncoder -> LogisticRegression` — **no scaler**, so coefficients live in WOE space and `beta * WOE` is the exact sub-score basis for Phase 2.

In [16]:
# === Step 5: WOEEncoder (formalized from woe_transform) ===

class WOEEncoder(BaseEstimator, TransformerMixin):
    """Apply WOE using a precomputed bin_definitions table.
    Numeric: value -> bin (via edges) -> WOE. Categorical: category -> WOE.
    Missing / unseen -> WOE 0 (neutral)."""
    def __init__(self, bin_definitions, features):
        self.bin_definitions = bin_definitions
        self.features = features

    def fit(self, X, y=None):
        self.numeric_, self.categorical_ = {}, {}
        for feat in self.features:
            d = (self.bin_definitions[self.bin_definitions['feature_name'] == feat]
                 .sort_values('bin_index'))
            if d['feature_type'].iloc[0] == 'numeric':
                edges = np.array([d['lower_bound'].iloc[0]] + d['upper_bound'].tolist(),
                                 dtype=float)
                self.numeric_[feat] = (edges, d['woe'].to_numpy())
            else:
                self.categorical_[feat] = dict(zip(d['category_value'].astype(str),
                                                   d['woe']))
        return self

    def transform(self, X):
        out = pd.DataFrame(index=X.index)
        for feat in self.features:
            if feat in self.numeric_:
                edges, woe_by_idx = self.numeric_[feat]
                idx = pd.cut(X[feat], bins=edges, labels=False, include_lowest=True)
                mapped = pd.Series(0.0, index=X.index)
                valid = idx.notna()
                mapped[valid] = woe_by_idx[idx[valid].astype(int).to_numpy()]
                out[feat] = mapped
            else:
                out[feat] = X[feat].astype(str).map(self.categorical_[feat]).fillna(0.0)
        return out.to_numpy()

# sanity: encoder must reproduce the prune's WOE matrix for the 12
_enc = WOEEncoder(bin_definitions, SELECTED_FEATURES).fit(df[SELECTED_FEATURES])
_check = woe_transform(df, bin_definitions, SELECTED_FEATURES)[SELECTED_FEATURES].to_numpy()
assert np.allclose(_enc.transform(df[SELECTED_FEATURES]), _check, equal_nan=True)
print("WOEEncoder reproduces the prune WOE matrix — OK")

WOEEncoder reproduces the prune WOE matrix — OK


In [17]:
# === Step 6: pipeline + plain-vs-balanced CV ===
X, y = df[SELECTED_FEATURES], df['TARGET']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def make_pipe(class_weight=None):
    return Pipeline([
        ('woe', WOEEncoder(bin_definitions, SELECTED_FEATURES)),
        ('lr', LogisticRegression(max_iter=1000, class_weight=class_weight)),
    ])

for name, cw in [('plain', None), ('balanced', 'balanced')]:
    auc = cross_val_score(make_pipe(cw), X, y, cv=cv, scoring='roc_auc')
    gini = 2 * auc - 1
    print(f"{name:9s}  AUC {auc.mean():.4f} ± {auc.std():.4f}   "
          f"Gini {gini.mean():.4f} ± {gini.std():.4f}")

plain      AUC 0.6829 ± 0.0030   Gini 0.3659 ± 0.0061
balanced   AUC 0.6830 ± 0.0030   Gini 0.3660 ± 0.0060


In [18]:
# === Section 6: fit final pipeline + build df_scored ===
X, y = df[SELECTED_FEATURES], df['TARGET']

pipe_plain = Pipeline([
    ('woe', WOEEncoder(bin_definitions, SELECTED_FEATURES)),
    ('lr',  LogisticRegression(max_iter=1000, class_weight=None)),
])

# honest CV number for the metadata file (same splits as Step 6)
_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc  = cross_val_score(pipe_plain, X, y, cv=_cv, scoring='roc_auc')
cv_gini = 2 * cv_auc - 1

pipe_plain.fit(X, y)   # full-train fit for scoring + pickle

df_scored = df[['SK_ID_CURR', 'TARGET']].copy()
df_scored['score'] = pipe_plain.predict_proba(X)[:, 1]

# DAYS_DECISION: recency (abs days) of the applicant's LAST previous application,
# median-filled where there's no prior app. Proxy decision date = the synthetic-period
# basis for Phase 2 PSI. CAVEAT: the no-prior-app cohort all lands on the median,
# so it clusters into a single period bucket downstream.
df_scored['DAYS_DECISION'] = df['prev_days_last_decision']

print(f"plain CV  AUC {cv_auc.mean():.4f} ± {cv_auc.std():.4f}   "
      f"Gini {cv_gini.mean():.4f} ± {cv_gini.std():.4f}")
print(df_scored.shape); print(df_scored.head())

plain CV  AUC 0.6829 ± 0.0030   Gini 0.3659 ± 0.0061
(307511, 4)
   SK_ID_CURR  TARGET     score  DAYS_DECISION
0      100002       1  0.137055          606.0
1      100003       0  0.018283          746.0
2      100004       0  0.060435          815.0
3      100006       0  0.067629          181.0
4      100007       0  0.029904          374.0


## 6. Export

In [19]:
df_scored.to_csv('scored_applications.csv', index=False)
print(f"score range: {df_scored['score'].min():.4f} – {df_scored['score'].max():.4f}")
print(df_scored.groupby('TARGET')['score'].describe().round(4))

score range: 0.0041 – 0.4802
           count    mean     std     min     25%     50%     75%     max
TARGET                                                                  
0       282686.0  0.0778  0.0493  0.0041  0.0427  0.0651  0.0998  0.4802
1        24825.0  0.1134  0.0642  0.0084  0.0655  0.0998  0.1474  0.4642


In [20]:
# === model_features.csv — WOE values, long, group-tagged ===
# Monitored set = model set (decision 5). feature_value = WOE (decision 6),
# so PSI on this == PSI on bin membership; the missing/neutral cohort is the WOE-0 bucket.
woe_vals = woe_transform(df, bin_definitions, SELECTED_FEATURES)

model_features = (
    df[['SK_ID_CURR']].join(woe_vals)
      .melt(id_vars='SK_ID_CURR', var_name='feature_name', value_name='feature_value')
)
model_features['feature_group'] = model_features['feature_name'].map(FEATURE_TO_GROUP)
assert model_features['feature_group'].notna().all(), "untagged feature(s)"

model_features.to_csv('model_features.csv', index=False)
print(f"model_features.csv: {model_features.shape}  "
      f"({model_features['feature_name'].nunique()} features, "
      f"{model_features['feature_group'].nunique()} groups)")

model_features.csv: (3690132, 4)  (12 features, 5 groups)


In [21]:
# === bin_definitions.csv — the production scorecard (the 12 model features' bins) ===
scorecard = bin_definitions[bin_definitions['feature_name'].isin(SELECTED_FEATURES)].copy()
scorecard.to_csv('bin_definitions.csv', index=False)
print(f"bin_definitions.csv: {scorecard.shape}  ({scorecard['feature_name'].nunique()} features)")
# drop the .isin(...) filter if you'd rather persist the full candidate pool for audit

bin_definitions.csv: (86, 14)  (12 features)


In [19]:
# === model_coefficients.csv — per-feature coefficients for Phase 2 pillar sub-scores ===
# Seed shape: feature_name, coefficient (NO group column — pillar lives in model_features,
# whose feature_group is the single source of truth for the taxonomy).
# Coefficients multiply WOE directly: the pipeline has no scaler, so beta * WOE is exact.
lr = pipe_plain.named_steps['lr']
model_coefficients = pd.DataFrame({
    'feature_name': SELECTED_FEATURES,   # WOEEncoder preserves this column order -> coef_ aligns positionally
    'coefficient':  lr.coef_[0],
})
model_coefficients.to_csv('model_coefficients.csv', index=False)
print(f"intercept (excluded from seed; constant shift, irrelevant to ranking): {lr.intercept_[0]:.6f}")
print(model_coefficients.to_string(index=False))

intercept (excluded from seed; constant shift, irrelevant to ranking): -2.438138
               feature_name  coefficient
         bur_avg_util_ratio     0.595927
        bur_avg_credit_days     0.237316
              DAYS_EMPLOYED     0.645467
                 DAYS_BIRTH     0.492726
     bur_newest_credit_days     0.501952
            OCCUPATION_TYPE     0.561615
            credit_to_goods     0.804121
           NAME_INCOME_TYPE     0.287979
           bur_avg_end_days     0.126267
          ORGANIZATION_TYPE     0.337038
REGION_RATING_CLIENT_W_CITY     0.904064
        NAME_EDUCATION_TYPE     0.775867


In [25]:
# === pickle + metadata (rewritten for the WOE path) ===
MODELS_DIR = Path.cwd() / "models"
MODELS_DIR.mkdir(exist_ok=True)
joblib.dump(pipe_plain, MODELS_DIR / "logistic_regression_woe_v1.pkl")

metadata = {
    "model_type":      "Pipeline(WOEEncoder -> LogisticRegression)",
    "feature_set":     "widened_woe_iv_selected_rare_collapsed",
    "n_features":      len(SELECTED_FEATURES),
    "feature_names":   list(SELECTED_FEATURES),
    "cv_auc":          float(cv_auc.mean()),
    "cv_auc_std":      float(cv_auc.std()),
    "gini":            float(cv_gini.mean()),
    "class_weight":    "none (plain)",
    "ext_source_used": False,
    "trained_on":      str(date.today()),
    "sklearn_version": sklearn.__version__,
    "python_version":  f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}",
}
with open(MODELS_DIR / "logistic_regression_woe_v1.json", "w") as f:
    json.dump(metadata, f, indent=2)
print(json.dumps(metadata, indent=2))

{
  "model_type": "Pipeline(WOEEncoder -> LogisticRegression)",
  "feature_set": "widened_woe_iv_selected_rare_collapsed",
  "n_features": 12,
  "feature_names": [
    "bur_avg_util_ratio",
    "bur_avg_credit_days",
    "DAYS_EMPLOYED",
    "DAYS_BIRTH",
    "bur_newest_credit_days",
    "OCCUPATION_TYPE",
    "credit_to_goods",
    "NAME_INCOME_TYPE",
    "bur_avg_end_days",
    "ORGANIZATION_TYPE",
    "REGION_RATING_CLIENT_W_CITY",
    "NAME_EDUCATION_TYPE"
  ],
  "cv_auc": 0.6829380538225914,
  "cv_auc_std": 0.0030425857877733114,
  "gini": 0.36587610764518264,
  "class_weight": "none (plain)",
  "ext_source_used": false,
  "trained_on": "2026-05-30",
  "sklearn_version": "1.8.0",
  "python_version": "3.13.7"
}


## 7. Gini Pool Calculation

In [2]:
PROCESSED = Path('data/processed')
df = pd.read_csv(PROCESSED / 'scored_applications.csv')



In [3]:
from sklearn.metrics import roc_auc_score
gini = 2 * roc_auc_score(df.TARGET, df.score) - 1
print(repr(gini))

0.36619854728484724
